In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_16_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 17 ###

# Optimized cell 17

def combine_subset_of_data_from_multiple_years_29(question, x_axis_title, include_2017=None):
    # mapping of mis-encoded degrees
    mapping = {
        "Masterâ\x80\x99s degree": "Master's degree",
        "Bachelorâ\x80\x99s degree": "Bachelor's degree"
    }
    # list of (year, dataframe) pairs
    year_dfs = [
        ('2022', responses_df_2022_cell10),
        ('2021', responses_df_2021),
        ('2020', responses_df_2020),
        ('2019', responses_df_2019_cell10),
        ('2018', responses_df_2018_cell10),
    ]
    if include_2017 is not None:
        year_dfs.append(('2017', responses_df_2017))

    records = []
    for year, df in year_dfs:
        # choose the correct column for 2017 vs. others
        col = question if question in df.columns else 'FormalEducation'
        s = df[col].replace(mapping)
        # compute percentages
        counts = s.value_counts(dropna=False)
        pct = (counts * 100 / counts.sum()).round(1)
        # turn into a DataFrame
        tmp = pct.sort_index().reset_index()
        tmp.columns = [x_axis_title, 'percentage']
        tmp['year'] = year
        records.append(tmp)

    return pd.concat(records, ignore_index=True)

# inputs
question_of_interest_cell29 = (
    'What is the highest level of formal education that you have attained or plan to attain within the next 2 years?'
)
title_for_x_axis_cell29 = ''
subset_of_degrees = ["Bachelor's degree", "Master's degree", 'Doctoral degree']

# combine, sort, then collapse all other categories into 'Other'
degree_df_combined_cell29 = (
    combine_subset_of_data_from_multiple_years_29(
        question_of_interest_cell29,
        title_for_x_axis_cell29,
        include_2017='yes'
    )
    .sort_values(['year', 'percentage'])
)
degree_df_combined_cell29[title_for_x_axis_cell29] = (
    degree_df_combined_cell29[title_for_x_axis_cell29]
    .where(
        degree_df_combined_cell29[title_for_x_axis_cell29].isin(subset_of_degrees),
        'Other'
    )
)

degree_df_combined_cell29.info()

<class 'pandas.core.frame.DataFrame'>
Index: 47 entries, 42 to 3
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0               47 non-null     object 
 1   percentage  47 non-null     float64
 2   year        47 non-null     object 
dtypes: float64(1), object(2)
memory usage: 1.5+ KB
CPU times: user 20.3 ms, sys: 40 µs, total: 20.3 ms
Wall time: 20.3 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_17_try_0.pickle